# **Testing for Sequential Linear Separability (MODULARIIZNG SLS)**

## **Imports**

In [1]:
import numpy as np
import tensorflow as tf
from sklearn.svm import LinearSVC, SVC
import matplotlib.pyplot as plt
import numpy as np
import itertools  # for isSLS

## **Function definitions**

In [2]:
def prepData(data):
    # Input: A tuple of numpy arrays with data and labels

    x_train, y_train = data[0], data[1].reshape(-1)
    x_train_flat = x_train.reshape(x_train.shape[0], -1)

    class_means = {}
    for label in np.unique(y_train):
        class_points = x_train_flat[y_train == label]

        # Compute mean vector
        mean_vector = np.mean(class_points, axis=0)
        class_means[label] = mean_vector

    # Returns the flattened data and the dictionary of class means
    return (x_train_flat, y_train), class_means

In [3]:
def isOrderedSLS(data, selected_classes, means):
    x_train, y_train = data[0], data[1]

    class_means = means

    # Check if strict subset of classes
    if len(selected_classes) < len(np.unique(y_train)):
        # Create filters
        # This returns a boolean array of the same size as y_train
        train_mask = np.isin(y_train, selected_classes)

        # Apply the filter to the training dataset
        x_train_subset = x_train[train_mask]
        y_train_subset = y_train[train_mask]
    else:
        x_train_subset = x_train
        y_train_subset = y_train

    ''' Algorithm '''
    # Combine class mean vectors into a matrix
    means_stack = np.stack([class_means[cls] for cls in selected_classes])

    # Compute the barycenter as the average of class means
    barycenter = np.mean(means_stack, axis=0)

    ''' From here on down I merged '''
    # Copies of the training data to modify throughout the experiment
    x_mod = x_train_subset.copy()
    y_mod = y_train_subset.copy()

    # Boolean of whether each class was perfectly separated
    separation_results = {}

    # Copy of class means to avoid mutation when collapsed
    class_means_copy = class_means.copy()

    # Collapse point (p_n) for each class
    pn_points = {}

    # Order of classes
    order = selected_classes
    ''' This seems unnecessary, make better later '''

    for n in order:
        # Exit if only one class remains in the data
        '''Not being implemented now because I removed the change in y_mod'''
        if len(np.unique(y_mod)) == 1:
            separation_results[n] = True
            # print(f"Classes {selected_classes} are linearly separable.")
            break

        # Binary labels: 1 for class n, -1 for all other classes
        y_binary = np.where(y_mod == n, 1, -1)

        # Training a linear SVM classifier (LinearSVC) to separate class n vs. the rest
        clf = LinearSVC(C=1.0, max_iter=10000)
        #clf = SVC(kernel='linear', C=1e6)
        clf.fit(x_mod, y_binary)

        # Evaluate performance through predictions
        preds = clf.predict(x_mod)

        # Error rate calculation for class n
        error_rate = np.mean(preds[y_binary == 1] != 1)

        # True if no misclassifications
        is_perfect = error_rate == 0.0

        # Results recorded for this class
        separation_results[n] = is_perfect

        #print(f"Perfect separation: {is_perfect}")

        # Condition for moving forward through the order
        if not is_perfect:
            print(f"Stopping due to separation failure at {n}. Classes {selected_classes} are not SLS in this order.")
            return False

        # SVM weight vector
        w = clf.coef_[0]

        # SVM bias
        b = clf.intercept_[0]

        # Mean of the current class
        mean_n = class_means_copy[n]

        # Vector pointing from class mean to barycenter
        direction = barycenter - mean_n

        # Solving for intersection (t)
        t = -(np.dot(w, mean_n) + b) / np.dot(w, direction)

        # Compute p_n using line equation
        p_n = mean_n + t * direction

        # Collapse point storage
        pn_points[n] = p_n

        # Replacing all class-n samples with the single point p_n
        x_mod[y_mod == n] = p_n

        #Removing all class-n entries from the dataset
        keep_mask = y_mod != n
        x_mod = x_mod[keep_mask]
        y_mod = y_mod[keep_mask]

    # These only happen if result is true and classes are indeed SLS
    print(f"Classes {selected_classes} are SLS.")
    return True
    #return separation_results, pn_points

In [4]:
def isSLS(data, selected_classes, means):
    # Computes class means to pass to isOrderedSLS, if necessary
    x_train, y_train = data[0], data[1]

    class_means = means

    # Iterates through every possible order in the selected_classes
    for order in itertools.permutations(selected_classes):
        if isOrderedSLS(data, order, class_means)==True:
            print(f"The selected classes are SLS, in the following order: {order}.")
            return True

    # Function continues if no isOrderedSLS returned True
    print("The selected classes are likely not SLS.")
    return False

## **Running Experiments with isOrderedSLS**

In [5]:
# Load data from TensorFlow
(x_train, y_train), _ = tf.keras.datasets.mnist.load_data()

In [6]:
data, means = prepData((x_train, y_train))

In [7]:
isOrderedSLS(data, [0,2,1], means)

Classes [0, 2, 1] are SLS.


True

## **Running Experiments with isSLS**

In [8]:
isSLS(data, [2,0,1], means)

Stopping due to separation failure at 2. Classes (2, 0, 1) are not SLS in this order.
Stopping due to separation failure at 2. Classes (2, 1, 0) are not SLS in this order.
Classes (0, 2, 1) are SLS.
The selected classes are SLS, in the following order: (0, 2, 1).


True

# **Fact 3.1. MNIST is not sequentially linearly seperable**

In [9]:
isOrderedSLS(data, [4,9], means)
isOrderedSLS(data, [3,5], means)

Stopping due to separation failure at 4. Classes [4, 9] are not SLS in this order.
Stopping due to separation failure at 3. Classes [3, 5] are not SLS in this order.


False

**Proof of Fact 3.1:** *Sequential linear separability requires that, at the final stage, the remaining two classes are linearly separable. Using our SLS testing procedure, we observe that the digit pairs (4,9) and (3,5) are not linearly separable. Therefore, no ordering of the full MNIST dataset can satisfy the SLS condition. Hence MNIST is not sequentially linearly separable.*